# Journal 2 - Scoring and baselines

**Question.** What must a model beat to be worth deploying?
**Answer.** B2 (TF-IDF + logistic regression): on `test_natural` it finds the right desk for 92 % of tickets, yet **34 tickets in 100 get a definite answer that is wrong** (a silent error), and 6.6 % of urgent tickets go to the wrong desk. It breaks on the extraction traps (missing currency, odd id formats).

How a ticket is judged (`docs/scoring_protocol.md`): the copilot routes it by itself when sure, and hands it to a human otherwise. A ticket handed to a human costs a little work. A **silent error** (routed to a desk with a wrong desk or a wrong reference) is the worst: nothing flags it and the draft is built on it. The error severity score (human = 1, wrong desk = 3, wrong reference = 5, urgent ticket in a wrong desk = 10 points; assumed weights) summarises it in one number.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks")]
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import _viz as viz
import _tracking
viz.setup()

from src import scoring
from src.baselines import predict_majority, predict_rules, fit_tfidf, predict_tfidf
print(scoring.DEFAULT_WEIGHTS)

## 1. Does the scorer behave? Two trivial systems on val
A perfect system must have no error at all; sending everything to a human must give no silent error, and a severity of 1 point for each ticket that has a desk (83 % of them), so 0.83.

In [ ]:
from src.dataset_build.schema import TicketLabel
val_x, val_y, _ = scoring.load_split(ROOT / "data/dataset/val.csv")
train_x, train_y, _ = scoring.load_split(ROOT / "data/dataset/train.csv")
perfect = [json.dumps(l.model_dump()) for l in val_y]
triage = json.dumps(TicketLabel(intent="OTHER", sub_intent="OTHER", transaction_id="", amount="", currency="",
                                urgency="UNDEFINED", sentiment="UNDEFINED").model_dump())
pd.DataFrame({"perfect": scoring.score(val_y, perfect), "all to a human": scoring.score(val_y, [triage] * len(val_y))}).loc[
    ["silent_error_rate", "needless_triage_rate", "auto_routed_rate", "desk_accuracy", "severity_score"]].round(3)

Both behave as designed, so the scale is trustworthy. Note that the perfect system routes only 83 % of the tickets by itself: the other 17 % are `OTHER` or `UNDEFINED` and belong to a human.

## 2. The three baselines on val
B0 majority class, B1 keywords + regexes, B2 TF-IDF + logistic regression (regularisation `C` chosen on val only).

In [ ]:
SHOWN = ["silent_error_rate", "needless_triage_rate", "urgent_wrong_desk", "auto_routed_rate", "desk_accuracy",
         "exact_transaction_id", "exact_amount", "severity_score"]
grid = {c: scoring.score(val_y, predict_tfidf(fit_tfidf(train_x, train_y, c), val_x))["severity_score"] for c in (0.3, 1.0, 3.0, 10.0)}
best_c = min(grid, key=grid.get)
val = {"B0": scoring.score(val_y, predict_majority(train_y, val_x)), "B1": scoring.score(val_y, predict_rules(val_x)),
       "B2": scoring.score(val_y, predict_tfidf(fit_tfidf(train_x, train_y, best_c), val_x))}
print("severity score by C:", {c: round(v, 3) for c, v in grid.items()}, "-> kept C =", best_c)
pd.DataFrame(val).loc[SHOWN].round(3)

B0 is the floor: it answers every ticket and is wrong on most. B1 hands a quarter of the tickets to a human and still gets the desk right only 40 % of the time. B2 is a real competitor (95 % desk accuracy) but a third of its definite answers carry a wrong desk or a wrong reference. The reference fields (id, amount) are extracted by the same regexes in B1 and B2: that is where a language model has room to win.

## 3. The baselines on `test_natural` (read once)
610 tickets, written by another model than train. Interval = 95 % bootstrap.

In [ ]:
nat_x, nat_y, nat_traps = scoring.load_split(ROOT / "data/dataset/test_natural.csv")
models = fit_tfidf(train_x, train_y, best_c)
preds = {"B0": predict_majority(train_y, nat_x), "B1": predict_rules(nat_x), "B2": predict_tfidf(models, nat_x)}
b3 = _tracking.load_predictions(ROOT, "B3", "test_natural", None, nat_x)
if b3 is not None:
    preds["B3"] = b3
res = {k: scoring.score(nat_y, v) for k, v in preds.items()}
ci = {k: scoring.mean_interval(scoring.silent_errors(nat_y, v)) for k, v in preds.items()}
display(pd.DataFrame(res).loc[SHOWN].round(3))

fig, ax = plt.subplots(figsize=(7.5, 2.8))
viz.hbar(ax, [{"B0": "B0 majority class", "B1": "B1 keywords + regex", "B2": "B2 TF-IDF + LR", "B3": "B3 prompted base model"}[k] for k in preds], [100 * ci[k][0] for k in preds],
         [viz.SYSTEM_COLORS[k] for k in preds], fmt="{:.1f}", errors=[(100 * ci[k][1], 100 * ci[k][2]) for k in preds])
ax.set_xlim(0, 135); ax.set_xlabel("silent errors per 100 tickets (lower is better)")
viz.headline(fig, "Silent errors by baseline", "test_natural • 610 tickets • 95% interval")
viz.save(fig, "02_baselines_silent")
plt.show()

B2 > B1 > B0 in quality, as expected, but even B2 routes 6.6 % of the urgent tickets to the wrong desk (11 of 167), the costliest error. Error severity score: B2 1.67, B1 2.27, B0 3.89 points.

## 4. The traps: where does B2 break?
`test_traps` has 23 trap families (380 tickets). Counts only below 20 tickets, no interval.

In [ ]:
trap_x, trap_y, trap_tags = scoring.load_split(ROOT / "data/dataset/test_traps.csv")
b2 = pd.DataFrame(scoring.silent_by_trap(trap_y, predict_tfidf(models, trap_x), trap_tags), index=["n", "rate"]).T
top = b2.sort_values("rate", ascending=False).head(12)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
viz.hbar(ax, [f"{k}  (n={int(n)})" for k, n in zip(top.index, top["n"])], (100 * top["rate"]).tolist(),
         [viz.ALERT if r >= 0.8 else viz.SYSTEM_COLORS["B2"] for r in top["rate"]], fmt="{:.0f}")
ax.set_xlabel("B2 silent errors per 100 tickets")
viz.headline(fig, "B2 silent errors by trap family", "test_traps • 12 highest silent-error families • n shown")
viz.save(fig, "02_b2_traps")
plt.show()
print("trap families where B2 has at least 80 silent errors per 100:", int((b2["rate"] >= 0.8).sum()), "of", len(b2))

B2 still finds the right desk (accuracy mostly 0.9-1.0) but fails to extract the reference: missing currency, lowercase or spaced ids, two amounts. A model that *reads* the ticket should fix exactly this.

**Bar for the next journal: fewer than 34 silent errors per 100 tickets and fewer than 6.6 % of urgent tickets in a wrong desk.**

## 5. Base model versus fine-tuning on validationB3 is the prompted Qwen2.5-3B base model; D uses the same base model after LoRA fine-tuning. Predictions load from MLflow when available and otherwise from the locally validated exports. This is a validation comparison only; it does not reopen either final test split.

In [ ]:
fallbacks = {'B3': ROOT / 'data/runs/base_val_predictions.csv',
             'D': ROOT / 'docs/results/d_qwen2.5-3b_full_run_val_predictions.csv'}
val_compare = {'B2': val['B2']}
val_flags = {'B2': scoring.silent_errors(val_y, predict_tfidf(fit_tfidf(train_x, train_y, best_c), val_x))}
for system, fallback in fallbacks.items():
    predictions = _tracking.load_predictions(ROOT, system, 'val', fallback, val_x)
    if predictions is None:
        print(f'{system}/val unavailable; omitted')
        continue
    val_compare[system] = scoring.score(val_y, predictions)
    val_flags[system] = scoring.silent_errors(val_y, predictions)
display(pd.DataFrame(val_compare).loc[['silent_error_rate','urgent_wrong_desk','severity_score','desk_accuracy','auto_routed_rate']].round(3))
order = [system for system in ('B2','B3','D') if system in val_compare]
intervals = {system: scoring.mean_interval(val_flags[system]) for system in order}
fig, ax = plt.subplots(figsize=(7.5, 3.2))
viz.hbar(ax, order, [100*intervals[s][0] for s in order], [viz.SYSTEM_COLORS[s] for s in order],
         fmt='{:.1f}', errors=[(100*intervals[s][1],100*intervals[s][2]) for s in order])
ax.set_xlabel('Silent errors per 100 validation tickets · 95% interval')
viz.headline(fig, 'Silent errors on validation', 'B2, B3 and D · same 293 tickets')
viz.save(fig, '02_val_base_vs_finetuned')
plt.show()